# MGnify List vs. Detail endpoints

The [MGnify API](https://www.ebi.ac.uk/metagenomics/api/v2) has 2 types of endpoints: 

1. **list** endpoints which return a (paginated) list of records (dicts) in brief from a MGnify resource
2. **detail** endpoints which return a single record (dict) in lots of detail

The list endpoints can accept different search params to filter down the list (e.g., "search", "biome_lineage", "page_size"). In contrast, the detail endpoints only accept a single accession/id. 

In MGni.py, the MGnifier's that correspond to 
1. list endpoints are plural e.g. `MG.samples`
2. detail endpoints are singular e.g. `MG.sample`

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
from mgnipy import MGnipy

# init client
MG = MGnipy(cache_dir=None)

# check out the endpoints
print(MG.list_resources())

['analyses', 'analysis', 'assemblies', 'assembly', 'genomes', 'genome', 'publications', 'publication', 'samples', 'sample', 'studies', 'study', 'runs', 'run', 'biomes', 'biome', 'miscellaneous', 'catalogues', 'catalogue', 'private_studies']


From `list_resources()` we see plural vs. singular terms e.g. `analyses` vs. `analysis`. 

The plural attributes are list and singular are detail endpoints:

In [5]:
# accessing a list endpoint
studies_list = MG.studies(search='diabetes')
print(studies_list)
# now getting the list
with MG: 
    studies_list.get() # or .get_all()

display(studies_list.search_results.to_pandas().head())

# accessing a detail endpoint
a_study_detail = MG.study("MGYS00006805")
print(a_study_detail)
# now getting the record
with MG: 
    a_study_detail.get()

display(a_study_detail.search_results.to_pandas())

MGnifier instance for resource: studies
I.e., mgnipy.V2.proxies.studies.Studies
----------------------------------------
Base URL: https://www.ebi.ac.uk/
Parameters: {'search': 'diabetes'}
Example request URL: https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=diabetes&page=1
Endpoint module: mgnipy.emgapi_v2_client.api.studies.list_mgnify_studies
Is list endpoint (returns paginated results): True
Cache directory: None



,accession,ena_accessions,title,biome,updated_at,metadata
0,MGYS00006805,"[PRJEB63337, ERP148499]",EMG produced TPA metagenomics assembly of PRJD...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-01T19:06:56.381000+00:00,{}
1,MGYS00010387,"[PRJDB9649, DRP008416]",Fecal microbiota transplantation alleviates di...,None,2026-05-28T15:46:48.985000+00:00,{}
2,MGYS00010379,"[SRP387956, PRJNA862077]",gut metagenome and type-1 diabetes,None,2026-05-28T15:46:49.164000+00:00,{}
3,MGYS00005377,"[ERP114158, PRJEB31588]",EMG produced TPA metagenomics assembly of the ...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:46:57.525000+00:00,{}
4,MGYS00005378,"[SRP056054, PRJNA231909]","A prospective, longitudinal analysis of the de...","{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:46:57.527000+00:00,{}


MGnifier instance for resource: study
I.e., mgnipy.V2.proxies.studies.StudyDetail
----------------------------------------
Base URL: https://www.ebi.ac.uk/
Parameters: {'accession': 'MGYS00006805'}
Example request URL: https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00006805
Endpoint module: mgnipy.emgapi_v2_client.api.studies.get_mgnify_study
Is list endpoint (returns paginated results): False
Cache directory: None



,accession,ena_accessions,title,biome,updated_at,metadata,downloads,first_accession
0,MGYS00006805,"[PRJEB63337, ERP148499]",EMG produced TPA metagenomics assembly of PRJD...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-01T19:06:56.381000+00:00,{},"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP148499


## From list to detailed list

After getting a list of records from a list endpoint, one can beef up the list with additional metadata by using the "child" detail endpoint: e.g. `samples` to `sample`, `assemblies` to `assembly`, etc

there is a method `.enrich_details()` for MGnifyList's that help with this

In [ ]:
# populating the list 
with MG: 
    studies_list.enrich_details(limit=3) #can set to None to get all

# checking out the detailed metdata
studies_list.metadata.to_pandas(expand_nested_dicts=True)

Enriching study details:  50%|█████     | 6/12 [00:00<00:01,  5.02it/s]


,accession,ena_accessions,title,updated_at,downloads,first_accession,biome__biome_name,biome__lineage
0,MGYS00006805,"[PRJEB63337, ERP148499]",EMG produced TPA metagenomics assembly of PRJD...,2026-05-01T19:06:56.381000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP148499,Fecal,root:Host-associated:Human:Digestive system:La...
1,MGYS00010387,"[PRJDB9649, DRP008416]",Fecal microbiota transplantation alleviates di...,2026-05-28T15:46:48.985000+00:00,[],DRP008416,NaN,NaN
2,MGYS00010379,"[SRP387956, PRJNA862077]",gut metagenome and type-1 diabetes,2026-05-28T15:46:49.164000+00:00,[],SRP387956,NaN,NaN
3,MGYS00005377,"[ERP114158, PRJEB31588]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:46:57.525000+00:00,[],ERP114158,Fecal,root:Host-associated:Human:Digestive system:La...
4,MGYS00005378,"[SRP056054, PRJNA231909]","A prospective, longitudinal analysis of the de...",2026-05-28T15:46:57.527000+00:00,[],SRP056054,Fecal,root:Host-associated:Human:Digestive system:La...


## from detail to lists

PICK UP HERE

there are supported relationships